# End-to-End Retail Data Engineering Pipeline

## Overview

This notebook presents a complete retail analytics engineering workflow built with synthetic data. It generates source data, validates and cleans it, engineers warehouse-ready features, loads a DuckDB warehouse, runs SQL analysis, and publishes executive-level reporting artifacts.

The workflow is organized as a production-style pipeline:

Raw Data -> Validation -> Cleaning -> Feature Engineering -> Star Schema -> DuckDB Warehouse -> SQL Analytics -> Business KPIs -> Interactive Dashboard -> Data Quality Report -> Executive Summary

## Scale

- 10,000 customers
- 1,000 products
- 50 stores
- 100,000 orders

## Technologies

- Python
- Pandas
- NumPy
- Faker
- DuckDB
- Plotly
- PyArrow
- ReportLab

## Skills Demonstrated

- Data Engineering
- ETL
- Data Validation
- Data Cleaning
- Feature Engineering
- Dimensional Modeling
- DuckDB
- SQL Analytics
- Business Intelligence
- Dashboard Development
- Automated Reporting

In [ ]:
# ============================================================
# Imports
# ============================================================

# Core Python utilities
import random
import warnings
import logging

from pathlib import Path
from datetime import datetime, timedelta

# Data and analytics libraries
import duckdb
import numpy as np
import pandas as pd

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go

# Synthetic data generation
from faker import Faker

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

logger = logging.getLogger(__name__)

In [ ]:
# ============================================================
# Configuration
# ============================================================

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
Faker.seed(SEED)

fake = Faker()

# Synthetic dataset sizes
NUM_CUSTOMERS = 10000
NUM_PRODUCTS = 1000
NUM_STORES = 50
NUM_ORDERS = 100000

# Project paths
BASE_PATH = Path(".")

RAW_PATH = BASE_PATH / "data/raw"
CLEAN_PATH = BASE_PATH / "data/cleaned"
WAREHOUSE_PATH = BASE_PATH / "data/warehouse"
REPORT_PATH = BASE_PATH / "reports"

for folder in [
    RAW_PATH,
    CLEAN_PATH,
    WAREHOUSE_PATH,
    REPORT_PATH
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [199]:
REGIONS = [
    "Northeast",
    "Southeast",
    "Midwest",
    "Southwest",
    "West"
]

PAYMENT_METHODS = [
    "Credit Card",
    "Debit Card",
    "Apple Pay",
    "Google Pay",
    "PayPal",
    "Cash"
]

ORDER_STATUS = [
    "Completed",
    "Completed",
    "Completed",
    "Completed",
    "Completed",
    "Returned",
    "Cancelled"
]

SHIPPING_METHODS = [
    "Ground",
    "2-Day",
    "Overnight"
]

SEGMENTS = [
    "Consumer",
    "Corporate",
    "Small Business"
]

In [200]:
PRODUCT_CATALOG = {

    "Electronics": [
        "Laptop",
        "Phone",
        "Tablet",
        "Monitor",
        "Keyboard",
        "Mouse",
        "Headphones",
        "Camera"
    ],

    "Furniture": [
        "Desk",
        "Chair",
        "Standing Desk",
        "Bookshelf",
        "Cabinet"
    ],

    "Clothing": [
        "T-Shirt",
        "Jacket",
        "Jeans",
        "Sneakers",
        "Hat"
    ],

    "Home": [
        "Vacuum",
        "Coffee Maker",
        "Blender",
        "Microwave",
        "Lamp"
    ],

    "Sports": [
        "Basketball",
        "Football",
        "Bicycle",
        "Yoga Mat",
        "Tennis Racket"
    ]

}

In [201]:
def random_date():

    return fake.date_time_between(
        start_date="-2y",
        end_date="now"
    )


def random_margin():

    return round(
        random.uniform(1.20,2.30),
        2
    )


def random_discount():

    choices = [0,.05,.10,.15,.20]

    weights = [.55,.15,.15,.10,.05]

    return random.choices(
        choices,
        weights=weights,
        k=1
    )[0]

Generate Customers

In [202]:
customers = []

for customer_id in range(1, NUM_CUSTOMERS + 1):

    signup = fake.date_between("-5y", "today")

    customers.append({

        "CustomerID": customer_id,

        "CustomerName": fake.name(),

        "Email": fake.email(),

        "Phone": fake.phone_number(),

        "City": fake.city(),

        "State": fake.state(),

        "Region": random.choice(REGIONS),

        "Segment": random.choice(SEGMENTS),

        "SignupDate": signup

    })

customers = pd.DataFrame(customers)

customers.head()

,CustomerID,CustomerName,Email,Phone,City,State,Region,Segment,SignupDate
0,1,Donald Walker,rhodespatricia@example.org,001-260-501-3389,Robinsonshire,Louisiana,Northeast,Consumer,2024-09-29
1,2,Robert Wolfe,joshua35@example.org,361-855-9407,New Kellystad,Oregon,Midwest,Consumer,2025-08-05
2,3,Matthew Mejia,wyattmichelle@example.com,575-425-5341x928,Grayside,Iowa,Southeast,Consumer,2022-07-05
3,4,Patty Perez,trujillorichard@example.org,495.537.6724,Juliechester,Pennsylvania,Northeast,Small Business,2024-12-20
4,5,Zachary Hicks,camposmichelle@example.org,+1-326-691-6697x8480,Danielchester,Hawaii,West,Consumer,2023-07-18


Generate Products

In [203]:
products = []

brands = [
    "Nova",
    "Vertex",
    "Apex",
    "Fusion",
    "Atlas",
    "Pioneer",
    "Prime"
]

for product_id in range(1, NUM_PRODUCTS + 1):

    category = random.choice(
        list(PRODUCT_CATALOG.keys())
    )

    product = random.choice(
        PRODUCT_CATALOG[category]
    )

    cost = round(
        random.uniform(5,700),
        2
    )

    price = round(
        cost * random_margin(),
        2
    )

    products.append({

        "ProductID": product_id,

        "Category": category,

        "Brand": random.choice(brands),

        "ProductName": product,

        "Cost": cost,

        "Price": price

    })

products = pd.DataFrame(products)

products.head()

,ProductID,Category,Brand,ProductName,Cost,Price
0,1,Electronics,Fusion,Camera,41.86,59.86
1,2,Sports,Prime,Tennis Racket,392.99,738.82
2,3,Electronics,Prime,Tablet,92.41,117.36
3,4,Electronics,Atlas,Phone,606.35,897.40
4,5,Electronics,Prime,Laptop,186.82,375.51


Generate Stores

In [204]:
stores = []

for store_id in range(1, NUM_STORES + 1):

    stores.append({

        "StoreID": store_id,

        "StoreName": f"Store {store_id}",

        "City": fake.city(),

        "State": fake.state(),

        "Region": random.choice(REGIONS),

        "Manager": fake.name(),

        "OpenDate": fake.date_between(
            "-15y",
            "-1y"
        )

    })

stores = pd.DataFrame(stores)

stores.head()

,StoreID,StoreName,City,State,Region,Manager,OpenDate
0,1,Store 1,South Lauraside,South Dakota,West,William Castro,2021-12-16
1,2,Store 2,East Stevenshire,Utah,Northeast,Renee Thomas,2012-11-09
2,3,Store 3,Lake Aaron,New York,Midwest,Kristopher Henry,2020-02-13
3,4,Store 4,Nealmouth,South Carolina,Midwest,Michael Garcia,2016-01-31
4,5,Store 5,Richardhaven,Kentucky,West,Danielle Weaver,2014-08-06


 Generate Retail Orders

The order dataset simulates two years of retail transactions.

The simulation includes:

- Seasonal purchasing
- Black Friday and Cyber Monday spikes
- Holiday shopping increases
- Repeat customers
- Different purchasing behaviors by customer segment
- Discounts
- Shipping methods
- Payment methods
- Returns
- Cancellations

In [205]:
# ======================================================
# Major Shopping Events
# ======================================================

BLACK_FRIDAY = [

    datetime(2024,11,29).date(),

    datetime(2025,11,28).date()

]

CYBER_MONDAY = [

    datetime(2024,12,2).date(),

    datetime(2025,12,1).date()

]

CHRISTMAS = [

    datetime(2024,12,25).date(),

    datetime(2025,12,25).date()

]

In [206]:
# ======================================================
# Customer Spending Behavior
# ======================================================

segment_quantity = {

    "Consumer":(1,3),

    "Corporate":(2,10),

    "Small Business":(1,6)

}

segment_discount = {

    "Consumer":0.05,

    "Corporate":0.12,

    "Small Business":0.08

}

In [207]:
MONTH_MULTIPLIER = {

    1:0.85,

    2:0.90,

    3:1.00,

    4:1.02,

    5:1.05,

    6:1.10,

    7:1.15,

    8:1.08,

    9:1.02,

    10:1.10,

    11:1.45,

    12:1.60

}

In [208]:
customer_lookup = customers.set_index(
    "CustomerID"
)

product_lookup = products.set_index(
    "ProductID"
)

store_lookup = stores.set_index(
    "StoreID"
)

In [209]:
orders = []

start_date = datetime(2024,1,1)

end_date = datetime(2025,12,31)

date_range = (end_date-start_date).days

for order_id in range(1,NUM_ORDERS+1):

    customer_id = random.randint(
        1,
        NUM_CUSTOMERS
    )

    customer = customer_lookup.loc[
        customer_id
    ]

    product_id = random.randint(
        1,
        NUM_PRODUCTS
    )

    product = product_lookup.loc[
        product_id
    ]

    store_id = random.randint(
        1,
        NUM_STORES
    )

    order_date = start_date + timedelta(
        days=random.randint(
            0,
            date_range
        )
    )

    quantity = random.randint(
        *segment_quantity[
            customer["Segment"]
        ]
    )

    discount = random_discount()

    base_sales = quantity * product["Price"]

    sales = base_sales*(1-discount)

    cost = quantity*product["Cost"]

    profit = sales-cost

    shipping = random.choice(
        SHIPPING_METHODS
    )

    payment = random.choice(
        PAYMENT_METHODS
    )

    status = random.choice(
        ORDER_STATUS
    )

    orders.append({

        "OrderID":order_id,

        "CustomerID":customer_id,

        "ProductID":product_id,

        "StoreID":store_id,

        "OrderDate":order_date,

        "Quantity":quantity,

        "UnitPrice":product["Price"],

        "UnitCost":product["Cost"],

        "Discount":discount,

        "Sales":round(sales,2),

        "Profit":round(profit,2),

        "ShippingMethod":shipping,

        "PaymentMethod":payment,

        "Status":status

    })

orders = pd.DataFrame(
    orders
)

orders.head()

,OrderID,CustomerID,ProductID,StoreID,OrderDate,Quantity,UnitPrice,UnitCost,Discount,Sales,Profit,ShippingMethod,PaymentMethod,Status
0,1,2203,662,21,2024-12-23,1,1571.40,698.40,0.0,1571.40,873.00,2-Day,Credit Card,Completed
1,2,968,707,34,2024-12-01,3,288.22,132.21,0.1,778.19,381.56,Overnight,Cash,Completed
2,3,700,291,21,2024-03-03,1,705.05,437.92,0.0,705.05,267.13,Overnight,Google Pay,Completed
3,4,6562,72,22,2025-12-14,10,611.89,271.95,0.0,6118.90,3399.40,2-Day,Google Pay,Completed
4,5,7207,81,14,2025-12-07,3,665.62,426.68,0.0,1996.86,716.82,Overnight,Google Pay,Completed


In [210]:
holiday_mask = (

    orders["OrderDate"].isin(BLACK_FRIDAY)

    |

    orders["OrderDate"].isin(CYBER_MONDAY)

)

orders.loc[holiday_mask,"Sales"] *= 1.40

orders.loc[holiday_mask,"Profit"] *= 1.40

In [211]:
orders["Weekday"] = pd.to_datetime(
    orders["OrderDate"]
).dt.dayofweek

weekend = orders["Weekday"]>=5

orders.loc[
    weekend,
    "Sales"
] *= 1.10

orders.loc[
    weekend,
    "Profit"
] *= 1.10

In [212]:
orders["Month"] = pd.to_datetime(
    orders["OrderDate"]
).dt.month

orders["SeasonFactor"] = orders[
    "Month"
].map(
    MONTH_MULTIPLIER
)

orders["Sales"] *= orders["SeasonFactor"]

orders["Profit"] *= orders["SeasonFactor"]

In [213]:
shipping_days = {

    "Ground":5,

    "2-Day":2,

    "Overnight":1

}

orders["ShippingDays"] = orders[
    "ShippingMethod"
].map(
    shipping_days
)

orders["ShipDate"] = (

    pd.to_datetime(
        orders["OrderDate"]
    )

    +

    pd.to_timedelta(
        orders["ShippingDays"],
        unit="D"
    )

)

In [214]:
returns = orders.sample(

    frac=.07,

    random_state=42

).copy()

reasons = [

    "Damaged",

    "Wrong Item",

    "Changed Mind",

    "Late Delivery",

    "Defective"

]

returns["ReturnDate"] = (

    returns["ShipDate"]

    +

    pd.to_timedelta(

        np.random.randint(

            2,

            20,

            len(returns)

        ),

        unit="D"

    )

)

returns["Reason"] = np.random.choice(

    reasons,

    len(returns)

)

returns = returns[

    [

        "OrderID",

        "ReturnDate",

        "Reason"

    ]

]

returns.head()

,OrderID,ReturnDate,Reason
75721,75722,2025-03-05,Damaged
80184,80185,2025-01-24,Changed Mind
19864,19865,2025-09-19,Late Delivery
76699,76700,2025-04-10,Wrong Item
92991,92992,2025-07-19,Changed Mind


In [215]:
# Duplicate Orders

duplicates = orders.sample(

    500,

    random_state=42

)

orders = pd.concat(

    [

        orders,

        duplicates

    ],

    ignore_index=True

)

In [216]:
# Missing Customer IDs

idx = orders.sample(

    frac=.01,

    random_state=1

).index

orders.loc[

    idx,

    "CustomerID"

] = np.nan

In [217]:
# Missing Sales

idx = orders.sample(

    frac=.005,

    random_state=2

).index

orders.loc[

    idx,

    "Sales"

] = np.nan

In [218]:
# Impossible Quantity

idx = orders.sample(

    frac=.003,

    random_state=3

).index

orders.loc[

    idx,

    "Quantity"

] = -2

In [219]:
# Future Dates

idx = orders.sample(

    frac=.002,

    random_state=4

).index

orders.loc[

    idx,

    "OrderDate"

] = datetime(

        2030,

        1,

        1

)

In [220]:
# Negative Profit Outliers

idx = orders.sample(

    frac=.002,

    random_state=5

).index

orders.loc[

    idx,

    "Profit"

] *= -8

In [221]:
customers.to_csv(

    RAW_PATH/"customers.csv",

    index=False

)

products.to_csv(

    RAW_PATH/"products.csv",

    index=False

)

stores.to_csv(

    RAW_PATH/"stores.csv",

    index=False

)

orders.to_csv(

    RAW_PATH/"orders.csv",

    index=False

)

returns.to_csv(

    RAW_PATH/"returns.csv",

    index=False

)

print("Raw data exported.")

Raw data exported.


# Data Profiling & Validation

This section profiles the raw datasets before any transformation work begins. It captures data types, missing values, duplicate records, unique values, memory usage, outliers, and business rule violations so the cleaning stage starts from a clear quality baseline.

In [222]:
# ============================================================
# Load Raw Data
# ============================================================

customers = pd.read_csv(
    RAW_PATH / "customers.csv",
    parse_dates=["SignupDate"]
)

products = pd.read_csv(
    RAW_PATH / "products.csv"
)

stores = pd.read_csv(
    RAW_PATH / "stores.csv",
    parse_dates=["OpenDate"]
)

orders = pd.read_csv(
    RAW_PATH / "orders.csv",
    parse_dates=[
        "OrderDate",
        "ShipDate"
    ]
)

returns = pd.read_csv(
    RAW_PATH / "returns.csv",
    parse_dates=["ReturnDate"]
)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [223]:
datasets = {
    "Customers": customers,
    "Products": products,
    "Stores": stores,
    "Orders": orders,
    "Returns": returns
}

In [224]:
def profile_dataset(df, name):

    profile = {

        "Dataset": name,

        "Rows": len(df),

        "Columns": len(df.columns),

        "Memory_MB": round(
            df.memory_usage(deep=True).sum()/1024**2,
            2
        ),

        "DuplicateRows": int(df.duplicated().sum()),

        "MissingCells": int(df.isna().sum().sum())

    }

    return profile

In [225]:
profiles = []

for name, dataframe in datasets.items():

    profiles.append(
        profile_dataset(
            dataframe,
            name
        )
    )

profile_df = pd.DataFrame(
    profiles
)

profile_df

,Dataset,Rows,Columns,Memory_MB,DuplicateRows,MissingCells
0,Customers,10000,9,4.28,0,0
1,Products,1000,6,0.18,0,0
2,Stores,50,7,0.01,0,0
3,Orders,100500,19,28.64,470,1507
4,Returns,7000,3,0.50,0,0


In [226]:
missing_summary = []

for name, df in datasets.items():

    missing = df.isna().sum()

    for column in missing.index:

        if missing[column] > 0:

            missing_summary.append({

                "Dataset": name,

                "Column": column,

                "Missing": missing[column],

                "Percent":

                    round(

                        missing[column] /

                        len(df)

                        *100,

                        2

                    )

            })

missing_df = pd.DataFrame(
    missing_summary
)

missing_df

,Dataset,Column,Missing,Percent
0,Orders,CustomerID,1005,1.0
1,Orders,Sales,502,0.5


In [227]:
duplicate_report = []

for name, df in datasets.items():

    duplicate_report.append({

        "Dataset": name,

        "Duplicates":

            df.duplicated().sum()

    })

duplicate_report = pd.DataFrame(
    duplicate_report
)

duplicate_report

,Dataset,Duplicates
0,Customers,0
1,Products,0
2,Stores,0
3,Orders,470
4,Returns,0


In [228]:
datatype_report = []

for name, df in datasets.items():

    for column in df.columns:

        datatype_report.append({

            "Dataset": name,

            "Column": column,

            "DataType":

                str(df[column].dtype)

        })

datatype_report = pd.DataFrame(
    datatype_report
)

datatype_report.head()

,Dataset,Column,DataType
0,Customers,CustomerID,int64
1,Customers,CustomerName,object
2,Customers,Email,object
3,Customers,Phone,object
4,Customers,City,object


Business Rule Validation

In [229]:
negative_quantity = orders[
    orders["Quantity"] <= 0
]

print(

    "Negative Quantity:",

    len(negative_quantity)

)

Negative Quantity: 302


In [230]:
missing_customer = orders[
    orders["CustomerID"].isna()
]

print(

    "Missing Customer IDs:",

    len(missing_customer)

)

Missing Customer IDs: 1005


In [231]:
future_orders = orders[

    orders["OrderDate"]

    >

    pd.Timestamp.today()

]

print(

    "Future Orders:",

    len(future_orders)

)

Future Orders: 201


In [232]:
negative_sales = orders[
    orders["Sales"] < 0
]

print(

    "Negative Sales:",

    len(negative_sales)

)

Negative Sales: 0


In [233]:
q1 = orders["Profit"].quantile(.25)

q3 = orders["Profit"].quantile(.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr

upper = q3 + 1.5 * iqr

profit_outliers = orders[

    (orders["Profit"] < lower)

    |

    (orders["Profit"] > upper)

]

len(profit_outliers)

7355

In [234]:
invalid_price = orders[

    orders["UnitPrice"]

    <

    orders["UnitCost"]

]

len(invalid_price)

0

In [235]:
validation_summary = pd.DataFrame({

    "Check":[

        "Duplicate Orders",

        "Missing Customer IDs",

        "Missing Sales",

        "Negative Quantity",

        "Future Dates",

        "Price < Cost",

        "Profit Outliers"

    ],

    "Issues":[

        orders.duplicated().sum(),

        orders.CustomerID.isna().sum(),

        orders.Sales.isna().sum(),

        len(negative_quantity),

        len(future_orders),

        len(invalid_price),

        len(profit_outliers)

    ]

})

validation_summary

,Check,Issues
0,Duplicate Orders,470
1,Missing Customer IDs,1005
2,Missing Sales,502
3,Negative Quantity,302
4,Future Dates,201
5,Price < Cost,0
6,Profit Outliers,7355


# Data Cleaning Pipeline

This section applies the quality rules identified during profiling, normalizes the source records, and prepares the cleaned datasets for warehouse modeling and reporting.

In [236]:
total_checks = validation_summary.shape[0]

failed = (

    validation_summary["Issues"] > 0

).sum()

quality_score = round(

    (1 - failed / total_checks)

    *100,

    2

)

print(

    f"Overall Data Quality Score: {quality_score}%"

)

Overall Data Quality Score: 14.29%


In [237]:
fig = px.bar(

    validation_summary,

    x="Check",

    y="Issues",

    title="Data Validation Issues",

    text="Issues"

)

fig.update_layout(

    xaxis_tickangle=-30

)

fig.show()

In [238]:
profile_df.to_csv(

    REPORT_PATH /

    "dataset_profile.csv",

    index=False

)

validation_summary.to_csv(

    REPORT_PATH /

    "validation_summary.csv",

    index=False

)

missing_df.to_csv(

    REPORT_PATH /

    "missing_values.csv",

    index=False

)

duplicate_report.to_csv(

    REPORT_PATH /

    "duplicates.csv",

    index=False

)

datatype_report.to_csv(

    REPORT_PATH /

    "data_types.csv",

    index=False
)

print("Validation reports exported.")

Validation reports exported.


# Feature Engineering & Dimensional Modeling

This section transforms the cleaned source tables into analysis-ready dimensions and facts. It also calculates customer, product, and store metrics that support reporting, segmentation, and dashboarding.

In [239]:
# ==========================================================
# Cleaning Audit Log
# ==========================================================

audit_log = []

def log_step(step,
             before,
             after,
             notes):

    audit_log.append({

        "Step":step,

        "Rows Before":before,

        "Rows After":after,

        "Rows Removed":before-after,

        "Notes":notes,

        "Timestamp":datetime.now()

    })

In [240]:
customers_clean = customers.copy()

products_clean = products.copy()

stores_clean = stores.copy()

orders_clean = orders.copy()

returns_clean = returns.copy()

Clean Customers

In [241]:
before = len(customers_clean)

customers_clean = customers_clean.drop_duplicates()

after = len(customers_clean)

log_step(

    "Remove Duplicate Customers",

    before,

    after,

    "Duplicate customer records removed."

)

In [242]:
customers_clean["Email"] = (

    customers_clean["Email"]

    .str.lower()

    .str.strip()

)

In [243]:
customers_clean["Phone"] = (

    customers_clean["Phone"]

    .astype(str)

    .str.replace(

        r"\D",

        "",

        regex=True

    )

)

In [244]:
before = len(customers_clean)

customers_clean = customers_clean[

    customers_clean.CustomerID.notna()

]

after = len(customers_clean)

log_step(

    "Missing Customer IDs",

    before,

    after,

    "Removed customers without IDs."

)

Clean Products

In [245]:
before = len(products_clean)

products_clean = (

    products_clean

    .drop_duplicates()

)

after = len(products_clean)

log_step(

    "Duplicate Products",

    before,

    after,

    "Removed duplicate products."

)

In [246]:
before = len(products_clean)

products_clean = products_clean[

    products_clean.Price >

    products_clean.Cost

]

after = len(products_clean)

log_step(

    "Price Validation",

    before,

    after,

    "Removed products priced below cost."

)

# DuckDB Data Warehouse

This stage loads the engineered tables into DuckDB and prepares the warehouse layer used for SQL analytics, KPI aggregation, and dashboard development.

In [247]:
before = len(orders_clean)

orders_clean = (

    orders_clean

    .drop_duplicates()

)

after = len(orders_clean)

log_step(

    "Duplicate Orders",

    before,

    after,

    "Duplicate orders removed."

)

In [248]:
before = len(orders_clean)

orders_clean = (

    orders_clean

    .dropna(

        subset=[

            "CustomerID"

        ]

    )

)

after = len(orders_clean)

log_step(

    "Missing Customers",

    before,

    after,

    "Orders without customers removed."

)

In [249]:
before = len(orders_clean)

orders_clean = (

    orders_clean

    .dropna(

        subset=[

            "Sales"

        ]

    )

)

after = len(orders_clean)

log_step(

    "Missing Sales",

    before,

    after,

    "Orders missing sales removed."

)

In [250]:
negative = (

    orders_clean.Quantity < 0

).sum()

orders_clean["Quantity"] = (

    orders_clean["Quantity"]

    .abs()

)

print(

    f"Corrected {negative} negative quantities."

)

Corrected 300 negative quantities.


In [251]:
before = len(orders_clean)

orders_clean = orders_clean[

    orders_clean.OrderDate

    <=

    pd.Timestamp.today()

]

after = len(orders_clean)

log_step(

    "Future Orders",

    before,

    after,

    "Removed future-dated transactions."

)

In [252]:
orders_clean["Sales"] = (

    orders_clean.Quantity

    *

    orders_clean.UnitPrice

    *

    (

        1 -

        orders_clean.Discount

    )

)

orders_clean["Sales"] = (

    orders_clean.Sales.round(2)

)

In [253]:
orders_clean["Profit"] = (

    orders_clean.Sales

    -

    (

        orders_clean.Quantity

        *

        orders_clean.UnitCost

    )

)

orders_clean["Profit"] = (

    orders_clean.Profit.round(2)

)

Remove Profit Outliers

In [254]:
Q1 = orders_clean.Profit.quantile(.25)

Q3 = orders_clean.Profit.quantile(.75)

IQR = Q3-Q1

lower = Q1-1.5*IQR

upper = Q3+1.5*IQR

before = len(orders_clean)

orders_clean = orders_clean[

    (orders_clean.Profit>=lower)

    &

    (orders_clean.Profit<=upper)

]

after = len(orders_clean)

log_step(

    "Profit Outliers",

    before,

    after,

    "Removed statistical outliers."

)

Data Type Standardization

In [255]:
orders_clean["CustomerID"] = (

    orders_clean.CustomerID

    .astype(int)

)

orders_clean["ProductID"] = (

    orders_clean.ProductID

    .astype(int)

)

orders_clean["StoreID"] = (

    orders_clean.StoreID

    .astype(int)

)

orders_clean["Quantity"] = (

    orders_clean.Quantity

    .astype(int)

)

Cleaning Summary

In [256]:
audit_df = pd.DataFrame(

    audit_log

)

audit_df

,Step,Rows Before,Rows After,Rows Removed,Notes,Timestamp
0,Remove Duplicate Customers,10000,10000,0,Duplicate customer records removed.,2026-07-20 09:32:45.591065
1,Missing Customer IDs,10000,10000,0,Removed customers without IDs.,2026-07-20 09:32:45.636080
2,Duplicate Products,1000,1000,0,Removed duplicate products.,2026-07-20 09:32:45.646444
3,Price Validation,1000,1000,0,Removed products priced below cost.,2026-07-20 09:32:45.653540
4,Duplicate Orders,100500,100030,470,Duplicate orders removed.,2026-07-20 09:32:45.722235
5,Missing Customers,100030,99025,1005,Orders without customers removed.,2026-07-20 09:32:45.745369
6,Missing Sales,99025,98529,496,Orders missing sales removed.,2026-07-20 09:32:45.830065
7,Future Orders,98529,98328,201,Removed future-dated transactions.,2026-07-20 09:32:45.885828
8,Profit Outliers,98328,91668,6660,Removed statistical outliers.,2026-07-20 09:32:45.944303


Export Clean Data

In [257]:
customers_clean.to_csv(

    CLEAN_PATH /

    "customers_clean.csv",

    index=False

)

products_clean.to_csv(

    CLEAN_PATH /

    "products_clean.csv",

    index=False

)

stores_clean.to_csv(

    CLEAN_PATH /

    "stores_clean.csv",

    index=False

)

orders_clean.to_csv(

    CLEAN_PATH /

    "orders_clean.csv",

    index=False

)

returns_clean.to_csv(

    CLEAN_PATH /

    "returns_clean.csv",

    index=False
)

In [258]:
customers_clean.to_parquet(

    CLEAN_PATH /

    "customers.parquet"

)

products_clean.to_parquet(

    CLEAN_PATH /

    "products.parquet"

)

stores_clean.to_parquet(

    CLEAN_PATH /

    "stores.parquet"

)

orders_clean.to_parquet(

    CLEAN_PATH /

    "orders.parquet"

)

returns_clean.to_parquet(

    CLEAN_PATH /

    "returns.parquet"
)

In [259]:
print("="*60)

print("Cleaning Pipeline Complete")

print("="*60)

print(f"Customers : {len(customers_clean):,}")

print(f"Products  : {len(products_clean):,}")

print(f"Stores    : {len(stores_clean):,}")

print(f"Orders    : {len(orders_clean):,}")

print(f"Returns   : {len(returns_clean):,}")

print("="*60)

Cleaning Pipeline Complete
Customers : 10,000
Products  : 1,000
Stores    : 50
Orders    : 91,668
Returns   : 7,000


# Feature Engineering & Dimensional Modeling

This section enriches the cleaned data with business-friendly features
and transforms the transactional dataset into a dimensional warehouse.

The final warehouse follows a Star Schema:

                    dim_customer
                          │
                          │
dim_store ───── fact_sales ───── dim_product
                          │
                          │
                      dim_date

The star schema improves query performance and simplifies reporting.

In [260]:
# ==========================================================
# Feature Engineering
# ==========================================================

fact_sales = orders_clean.copy()

Date Features

In [261]:
fact_sales["OrderDate"] = pd.to_datetime(
    fact_sales["OrderDate"]
)

fact_sales["ShipDate"] = pd.to_datetime(
    fact_sales["ShipDate"]
)

In [262]:
fact_sales["Year"] = fact_sales.OrderDate.dt.year

fact_sales["Quarter"] = fact_sales.OrderDate.dt.quarter

fact_sales["Month"] = fact_sales.OrderDate.dt.month

fact_sales["MonthName"] = fact_sales.OrderDate.dt.month_name()

fact_sales["Week"] = fact_sales.OrderDate.dt.isocalendar().week.astype(int)

fact_sales["Day"] = fact_sales.OrderDate.dt.day

fact_sales["Weekday"] = fact_sales.OrderDate.dt.day_name()

fact_sales["WeekendFlag"] = np.where(

    fact_sales.OrderDate.dt.dayofweek >= 5,

    1,

    0

)

Shipping Metrics

In [263]:
fact_sales["ShippingDays"] = (

    fact_sales["ShipDate"]

    -

    fact_sales["OrderDate"]

).dt.days

In [264]:
fact_sales["LateShipment"] = np.where(

    fact_sales["ShippingDays"] > 5,

    1,

    0

)

Financial Metrics

In [265]:
fact_sales["GrossRevenue"] = (

    fact_sales["Quantity"]

    *

    fact_sales["UnitPrice"]

)

In [266]:
fact_sales["DiscountAmount"] = (

    fact_sales["GrossRevenue"]

    *

    fact_sales["Discount"]

)

In [267]:
fact_sales["ProfitMargin"] = (

    fact_sales["Profit"]

    /

    fact_sales["Sales"]

)

fact_sales["ProfitMargin"] = (

    fact_sales["ProfitMargin"]

    .fillna(0)

    .round(4)

)

Customer Lifetime Value (CLV)

In [268]:
customer_revenue = (

    fact_sales

    .groupby("CustomerID")

    ["Sales"]

    .sum()

    .rename("CustomerLifetimeValue")

)

In [269]:
customers_clean = customers_clean.merge(

    customer_revenue,

    on="CustomerID",

    how="left"

)

customers_clean["CustomerLifetimeValue"] = (

    customers_clean["CustomerLifetimeValue"]

    .fillna(0)

)

Customer Purchase Frequency

In [270]:
purchase_frequency = (

    fact_sales

    .groupby("CustomerID")

    ["OrderID"]

    .count()

    .rename("PurchaseCount")

)

In [271]:
customers_clean = customers_clean.merge(

    purchase_frequency,

    on="CustomerID",

    how="left"

)

customers_clean["PurchaseCount"] = (

    customers_clean["PurchaseCount"]

    .fillna(0)

    .astype(int)

)

Average Order Value

In [272]:
aov = (

    fact_sales

    .groupby("CustomerID")

    ["Sales"]

    .mean()

    .rename("AverageOrderValue")

)

In [273]:
customers_clean = customers_clean.merge(

    aov,

    on="CustomerID",

    how="left"

)

customers_clean["AverageOrderValue"] = (

    customers_clean["AverageOrderValue"]

    .fillna(0)

)

RFM Analysis

In [274]:
#Recency
snapshot_date = (

    fact_sales["OrderDate"].max()

    +

    pd.Timedelta(days=1)

)

In [275]:
recency = (

    snapshot_date

    -

    fact_sales.groupby("CustomerID")

    ["OrderDate"]

    .max()

).dt.days.rename("Recency")

In [276]:
# Frequency
frequency = (

    fact_sales

    .groupby("CustomerID")

    ["OrderID"]

    .count()

    .rename("Frequency")
)

In [277]:
# Monetary 
monetary = (

    fact_sales

    .groupby("CustomerID")

    ["Sales"]

    .sum()

    .rename("Monetary")
)

In [278]:
rfm = pd.concat(

    [

        recency,

        frequency,

        monetary

    ],

    axis=1

)

rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
1,11,9,13293.06
2,61,11,13393.59
3,156,10,13023.21
4,371,5,7649.07
5,101,12,14584.53


Product Metrics

In [279]:
product_metrics = (

    fact_sales

    .groupby("ProductID")

    .agg(

        UnitsSold=("Quantity","sum"),

        Revenue=("Sales","sum"),

        Profit=("Profit","sum"),

        Orders=("OrderID","count")

    )

    .reset_index()

)

In [280]:
products_clean = products_clean.merge(

    product_metrics,

    on="ProductID",

    how="left"

)

Store Metrics

In [281]:
store_metrics = (

    fact_sales

    .groupby("StoreID")

    .agg(

        Revenue=("Sales","sum"),

        Profit=("Profit","sum"),

        Orders=("OrderID","count")

    )

    .reset_index()

)

In [282]:
stores_clean = stores_clean.merge(

    store_metrics,

    on="StoreID",

    how="left"
)

Date Dimension

In [283]:
dim_date = pd.DataFrame({

    "Date":pd.date_range(

        fact_sales.OrderDate.min(),

        fact_sales.OrderDate.max(),

        freq="D"

    )

})

In [284]:
dim_date["DateID"] = np.arange(

    1,

    len(dim_date)+1

)

dim_date["Year"] = dim_date.Date.dt.year

dim_date["Quarter"] = dim_date.Date.dt.quarter

dim_date["Month"] = dim_date.Date.dt.month

dim_date["MonthName"] = dim_date.Date.dt.month_name()

dim_date["Week"] = dim_date.Date.dt.isocalendar().week.astype(int)

dim_date["Day"] = dim_date.Date.dt.day

dim_date["Weekday"] = dim_date.Date.dt.day_name()

dim_date["Weekend"] = np.where(

    dim_date.Date.dt.dayofweek>=5,

    1,

    0

)

Add DateID

In [285]:
fact_sales = fact_sales.merge(

    dim_date[

        [

            "Date",

            "DateID"

        ]

    ],

    left_on="OrderDate",

    right_on="Date",

    how="left"

)

fact_sales.drop(

    columns="Date",

    inplace=True
)

Dimension Tables

In [286]:
dim_customer = customers_clean.copy()

dim_product = products_clean.copy()

dim_store = stores_clean.copy()

Final Fact Table

In [287]:
fact_sales = fact_sales[

    [

        "OrderID",

        "DateID",

        "CustomerID",

        "ProductID",

        "StoreID",

        "Quantity",

        "GrossRevenue",

        "Discount",

        "DiscountAmount",

        "Sales",

        "Profit",

        "ProfitMargin",

        "ShippingMethod",

        "PaymentMethod",

        "Status"

    ]

]

Warehouse Summary

In [288]:
print("="*60)

print("STAR SCHEMA CREATED")

print("="*60)

print(f"Fact Rows      : {len(fact_sales):,}")

print(f"Customers      : {len(dim_customer):,}")

print(f"Products       : {len(dim_product):,}")

print(f"Stores         : {len(dim_store):,}")

print(f"Dates          : {len(dim_date):,}")

print("="*60)

STAR SCHEMA CREATED
Fact Rows      : 91,668
Customers      : 10,000
Products       : 1,000
Stores         : 50
Dates          : 731


Export Warehouse Tables

In [289]:
dim_customer.to_parquet(

    WAREHOUSE_PATH /

    "dim_customer.parquet",

    index=False

)

dim_product.to_parquet(

    WAREHOUSE_PATH /

    "dim_product.parquet",

    index=False

)

dim_store.to_parquet(

    WAREHOUSE_PATH /

    "dim_store.parquet",

    index=False

)

dim_date.to_parquet(

    WAREHOUSE_PATH /

    "dim_date.parquet",

    index=False

)

fact_sales.to_parquet(

    WAREHOUSE_PATH /

    "fact_sales.parquet",

    index=False
)

# DuckDB Data Warehouse

DuckDB is an in-process analytical database optimized for OLAP workloads.

This section creates a complete analytical warehouse from the cleaned
dimensional model.

The warehouse contains:

- Fact Table
- Dimension Tables
- SQL Views
- Business Queries
- Performance Statistics

DuckDB allows us to query millions of rows with extremely fast execution
without requiring an external database server.

In [290]:
# ============================================================
# Export CSV Warehouse
# ============================================================

dim_customer.to_csv(
    WAREHOUSE_PATH/"dim_customer.csv",
    index=False
)

dim_product.to_csv(
    WAREHOUSE_PATH/"dim_product.csv",
    index=False
)

dim_store.to_csv(
    WAREHOUSE_PATH/"dim_store.csv",
    index=False
)

dim_date.to_csv(
    WAREHOUSE_PATH/"dim_date.csv",
    index=False
)

fact_sales.to_csv(
    WAREHOUSE_PATH/"fact_sales.csv",
    index=False
)

print("Warehouse CSV files exported.")

Warehouse CSV files exported.


In [291]:
# ============================================================
# Create Warehouse
# ============================================================

warehouse = duckdb.connect(

    WAREHOUSE_PATH/

    "retail_dw.duckdb"

)

print("Warehouse created.")

Warehouse created.


In [292]:
warehouse.execute("""

CREATE OR REPLACE TABLE dim_customer AS

SELECT *

FROM read_parquet(

'data/warehouse/dim_customer.parquet'

)

""")

In [293]:
warehouse.execute("""

CREATE OR REPLACE TABLE dim_product AS

SELECT *

FROM read_parquet(

'data/warehouse/dim_product.parquet'

)

""")

In [294]:
warehouse.execute("""

CREATE OR REPLACE TABLE dim_store AS

SELECT *

FROM read_parquet(

'data/warehouse/dim_store.parquet'

)

""")

In [295]:
warehouse.execute("""

CREATE OR REPLACE TABLE dim_date AS

SELECT *

FROM read_parquet(

'data/warehouse/dim_date.parquet'

)

""")

In [296]:
warehouse.execute("""

CREATE OR REPLACE TABLE fact_sales AS

SELECT *

FROM read_parquet(

'data/warehouse/fact_sales.parquet'

)

""")

Verify Tables

In [297]:
warehouse.execute("""

SHOW TABLES

""").df()

,name
0,dim_customer
1,dim_date
2,dim_product
3,dim_store
4,fact_sales
5,vw_customer_summary
6,vw_product_summary
7,vw_sales_summary
8,vw_store_summary


In [298]:
tables = [

    "fact_sales",

    "dim_customer",

    "dim_product",

    "dim_store",

    "dim_date"

]

for table in tables:

    rows = warehouse.execute(f"""

    SELECT COUNT(*)

    FROM {table}

    """).fetchone()[0]

    print(

        f"{table:<20}{rows:,}"

    )

fact_sales          91,668
dim_customer        10,000
dim_product         1,000
dim_store           50
dim_date            731


Build Analytical Views

In [299]:
warehouse.execute("""

CREATE OR REPLACE VIEW vw_sales_summary AS

SELECT

d.Year,

d.MonthName,

SUM(f.Sales) Revenue,

SUM(f.Profit) Profit,

COUNT(*) Orders

FROM fact_sales f

JOIN dim_date d

ON f.DateID=d.DateID

GROUP BY

d.Year,

d.MonthName

""")

In [300]:
warehouse.execute("""

CREATE OR REPLACE VIEW vw_customer_summary AS

SELECT

c.CustomerID,

c.CustomerName,

SUM(f.Sales) Revenue,

SUM(f.Profit) Profit,

COUNT(f.OrderID) Orders

FROM fact_sales f

JOIN dim_customer c

USING(CustomerID)

GROUP BY

ALL

""")

In [301]:
warehouse.execute("""

CREATE OR REPLACE VIEW vw_product_summary AS

SELECT

p.ProductName,

p.Category,

SUM(f.Sales) Revenue,

SUM(f.Profit) Profit,

SUM(f.Quantity) Units

FROM fact_sales f

JOIN dim_product p

USING(ProductID)

GROUP BY

ALL

""")

In [302]:
warehouse.execute("""

CREATE OR REPLACE VIEW vw_store_summary AS

SELECT

s.StoreName,

SUM(f.Sales) Revenue,

SUM(f.Profit) Profit,

COUNT(*) Orders

FROM fact_sales f

JOIN dim_store s

USING(StoreID)

GROUP BY

ALL

""")

Warehouse Statistics

In [303]:
warehouse.execute("""

SELECT

table_name,

estimated_size

FROM duckdb_tables()

""").df()

,table_name,estimated_size
0,dim_customer,10000
1,dim_date,731
2,dim_product,1000
3,dim_store,50
4,fact_sales,91668


Test Query

In [304]:
warehouse.execute("""

SELECT *

FROM vw_sales_summary

LIMIT 10

""").df()

,Year,MonthName,Revenue,Profit,Orders
0,2024,December,7053066.32,2657901.21,3880
1,2024,March,7155678.10,2700458.98,3889
2,2025,April,6988539.00,2612110.20,3745
3,2024,October,7172020.60,2686848.81,3906
4,2025,June,6991314.51,2637673.58,3838
5,2024,July,7270577.16,2724848.00,3928
6,2025,December,7058626.73,2681852.84,3899
7,2025,March,7065576.58,2629687.55,3824
8,2025,October,7305400.82,2714321.74,3950
9,2024,April,7076101.08,2647256.75,3869


Database Diagram

                 +-----------------+
                 |   dim_customer  |
                 +--------+--------+
                          |
                          |
                          |
+-------------+   +-------+-------+   +-------------+
| dim_store   |---|   fact_sales  |---| dim_product |
+-------------+   +-------+-------+   +-------------+
                          |
                          |
                 +--------+--------+
                 |    dim_date     |
                 +-----------------+

Warehouse Validation

In [305]:
warehouse.execute("""

SELECT

SUM(Sales) Revenue,

SUM(Profit) Profit,

COUNT(*) Orders

FROM fact_sales

""").df()

,Revenue,Profit,Orders
0,1.690323e+08,6.346416e+07,91668


Save Database

In [306]:

print("DuckDB Warehouse Saved.")

DuckDB Warehouse Saved.


# Advanced SQL Analytics

This section demonstrates analytical SQL techniques commonly used in
modern data engineering and analytics engineering roles.

Topics include:

- Window Functions
- Common Table Expressions (CTEs)
- Ranking
- Running Totals
- Rolling Averages
- Year-over-Year Growth
- Month-over-Month Growth
- Customer Segmentation
- Pareto Analysis
- Basket Analysis
- ABC Product Classification

In [307]:
# ======================================================
# SQL Helper
# ======================================================

def sql(query):

    return warehouse.execute(query).df()

In [308]:
sql("""

    SELECT

    SUM(Sales) Revenue,

    SUM(Profit) Profit,

    COUNT(*) Orders,

    AVG(Sales) AverageOrder

    FROM fact_sales

""")


,Revenue,Profit,Orders,AverageOrder
0,1.690323e+08,6.346416e+07,91668,1843.962353


In [309]:
sql("""

    SELECT

    Category,

    SUM(Sales) Revenue,

    SUM(f.Profit) Profit,

    SUM(Quantity) UnitsSold

    FROM fact_sales f

    JOIN dim_product p

    USING(ProductID)

    GROUP BY Category

    ORDER BY Revenue DESC

""")

,Category,Revenue,Profit,UnitsSold
0,Sports,39132115.86,14787563.22,72555.0
1,Furniture,35395144.30,13088255.44,67766.0
2,Electronics,35239898.47,13452612.18,66104.0
3,Clothing,30653051.84,11607554.93,61601.0
4,Home,28612130.47,10528174.17,60337.0


In [310]:
sql("""

SELECT

ProductName,

SUM(Sales) Revenue,

SUM(fact_sales.Profit) Profit

FROM fact_sales

JOIN dim_product

USING(ProductID)

GROUP BY ProductName

ORDER BY Revenue DESC

LIMIT 20

""")

,ProductName,Revenue,Profit
0,Tennis Racket,10198769.93,3977991.16
1,Desk,8378573.22,3101465.51
2,Cabinet,8237154.26,3062501.02
3,Bicycle,8156652.27,2938438.35
4,Chair,8151763.83,3081892.33
5,Yoga Mat,7803364.74,2948117.98
6,Jacket,7478878.18,2662150.24
7,Football,7098780.31,2730042.66
8,Hat,7075365.85,2793900.33
9,Phone,6917058.33,2396547.67


In [311]:
sql("""

SELECT

CustomerName,

SUM(Sales) Revenue,

COUNT(*) Orders

FROM fact_sales

JOIN dim_customer

USING(CustomerID)

GROUP BY CustomerName

ORDER BY Revenue DESC

LIMIT 20

""")

,CustomerName,Revenue,Orders
0,David Davis,120193.60,55
1,Cynthia Jones,118666.53,47
2,John Smith,110282.82,65
3,Eric Smith,98529.18,36
4,Thomas Johnson,83320.58,32
5,Michael Garcia,82292.29,52
6,Thomas Ross,79139.14,34
7,Timothy Johnson,77924.72,52
8,Christopher Smith,77258.15,41
9,Matthew Smith,76596.69,34


In [312]:
sql("""

SELECT

StoreName,

SUM(Sales) Revenue,

RANK()

OVER(

ORDER BY

SUM(Sales) DESC

)

StoreRank

FROM fact_sales

JOIN dim_store

USING(StoreID)

GROUP BY StoreName

""")

,StoreName,Revenue,StoreRank
0,Store 26,3556195.74,1
1,Store 35,3542212.95,2
2,Store 33,3535690.63,3
3,Store 7,3526650.59,4
4,Store 31,3518336.21,5
5,Store 44,3516738.53,6
6,Store 41,3514705.73,7
7,Store 47,3510257.74,8
8,Store 37,3502091.66,9
9,Store 9,3490916.81,10


In [313]:
sql("""

WITH revenue AS(

SELECT

DateID,

SUM(Sales) Revenue

FROM fact_sales

GROUP BY DateID

)

SELECT

DateID,

Revenue,

SUM(Revenue)

OVER(

ORDER BY DateID

)

RunningRevenue

FROM revenue

""")

,DateID,Revenue,RunningRevenue
0,1,246504.45,2.465044e+05
1,2,294879.84,5.413843e+05
2,3,226583.66,7.679679e+05
3,4,217312.08,9.852800e+05
4,5,238592.14,1.223872e+06
...,...,...,...
726,727,204196.26,1.681137e+08
727,728,234727.97,1.683484e+08
728,729,238268.44,1.685867e+08
729,730,197968.89,1.687847e+08


In [314]:
sql("""

WITH daily AS(

SELECT

DateID,

SUM(Sales) Revenue

FROM fact_sales

GROUP BY DateID

)

SELECT

DateID,

Revenue,

AVG(Revenue)

OVER(

ORDER BY DateID

ROWS BETWEEN

6 PRECEDING

AND CURRENT ROW

)

MovingAverage

FROM daily

""")

,DateID,Revenue,MovingAverage
0,1,246504.45,246504.450000
1,2,294879.84,270692.145000
2,3,226583.66,255989.316667
3,4,217312.08,246320.007500
4,5,238592.14,244774.434000
...,...,...,...
726,727,204196.26,214510.975714
727,728,234727.97,215324.040000
728,729,238268.44,221034.198571
729,730,197968.89,215412.778571


In [315]:
sql("""

WITH monthly AS(

SELECT

d.Year,

d.Month,

SUM(f.Sales) Revenue

FROM fact_sales f

JOIN dim_date d

ON f.DateID=d.DateID

GROUP BY

d.Year,

d.Month

)

SELECT

*,

Revenue

-

LAG(Revenue)

OVER(

ORDER BY

Year,

Month

)

Growth

FROM monthly

""")

,Year,Month,Revenue,Growth
0,2024,1,7441175.66,NaN
1,2024,2,6551995.37,-889180.29
2,2024,3,7155678.10,603682.73
3,2024,4,7076101.08,-79577.02
4,2024,5,7188799.20,112698.12
5,2024,6,7003817.51,-184981.69
6,2024,7,7270577.16,266759.65
7,2024,8,7010547.40,-260029.76
8,2024,9,7030107.23,19559.83
9,2024,10,7172020.60,141913.37


In [316]:
sql("""

WITH yearly AS(

SELECT

d.Year,

SUM(f.Sales) Revenue

FROM fact_sales f

JOIN dim_date d

USING(DateID)

GROUP BY d.Year

)

SELECT

*,

Revenue

-

LAG(Revenue)

OVER(

ORDER BY Year

)

YearGrowth

FROM yearly

""")

,Year,Revenue,YearGrowth
0,2024,8.475211e+07,NaN
1,2025,8.428023e+07,-471874.120001


In [317]:
sql("""

SELECT

CustomerName,

SUM(Sales) LifetimeValue

FROM fact_sales

JOIN dim_customer

USING(CustomerID)

GROUP BY CustomerName

ORDER BY LifetimeValue DESC

LIMIT 25

""")

,CustomerName,LifetimeValue
0,David Davis,120193.60
1,Cynthia Jones,118666.53
2,John Smith,110282.82
3,Eric Smith,98529.18
4,Thomas Johnson,83320.58
5,Michael Garcia,82292.29
6,Thomas Ross,79139.14
7,Timothy Johnson,77924.72
8,Christopher Smith,77258.15
9,Matthew Smith,76596.69


In [318]:
sql("""

SELECT

Segment,

COUNT(*) Customers,

AVG(CustomerLifetimeValue) AvgCLV

FROM dim_customer

GROUP BY Segment

""")

,Segment,Customers,AvgCLV
0,Corporate,3424,21810.377988
1,Consumer,3293,11557.109502
2,Small Business,3283,17147.744478


In [319]:
sql("""

WITH revenue AS(

SELECT

ProductID,

SUM(Sales) Revenue

FROM fact_sales

GROUP BY ProductID

),

ranked AS(

SELECT

*,

SUM(Revenue)

OVER(

ORDER BY Revenue DESC

)

RunningRevenue,

SUM(Revenue)

OVER()

TotalRevenue

FROM revenue

)

SELECT *

FROM ranked

""")

,ProductID,Revenue,RunningRevenue,TotalRevenue
0,604,370592.73,3.705927e+05,1.690323e+08
1,75,362420.70,7.330134e+05,1.690323e+08
2,340,359466.74,1.092480e+06,1.690323e+08
3,90,357739.52,1.450220e+06,1.690323e+08
4,855,349418.21,1.799638e+06,1.690323e+08
...,...,...,...,...
995,477,4065.20,1.690194e+08,1.690323e+08
996,319,3531.32,1.690229e+08,1.690323e+08
997,948,3308.28,1.690262e+08,1.690323e+08
998,605,3135.43,1.690294e+08,1.690323e+08


In [320]:
sql("""

WITH revenue AS(

SELECT

ProductID,

SUM(Sales) Revenue

FROM fact_sales

GROUP BY ProductID

),

ordered AS(

SELECT

*,

SUM(Revenue)

OVER(

ORDER BY Revenue DESC

)

RunningRevenue,

SUM(Revenue)

OVER()

TotalRevenue

FROM revenue

)

SELECT

ProductID,

Revenue,

CASE

WHEN

RunningRevenue/

TotalRevenue<=.8

THEN 'A'

WHEN

RunningRevenue/

TotalRevenue<=.95

THEN 'B'

ELSE 'C'

END

ABC

FROM ordered

""")

,ProductID,Revenue,ABC
0,604,370592.73,A
1,75,362420.70,A
2,340,359466.74,A
3,90,357739.52,A
4,855,349418.21,A
...,...,...,...
995,477,4065.20,C
996,319,3531.32,C
997,948,3308.28,C
998,605,3135.43,C


In [321]:
sql("""

SELECT

COUNT(*)

RepeatCustomers

FROM(

SELECT

CustomerID,

COUNT(*)

Orders

FROM fact_sales

GROUP BY CustomerID

HAVING Orders>=2

)

""")

,RepeatCustomers
0,9992


In [322]:
sql("""

SELECT

AVG(Quantity)

AverageBasket

FROM fact_sales

""")

,AverageBasket
0,3.58209


In [323]:
sql("""

SELECT

PaymentMethod,

COUNT(*) Orders

FROM fact_sales

GROUP BY PaymentMethod

ORDER BY Orders DESC

""")

,PaymentMethod,Orders
0,Google Pay,15473
1,Apple Pay,15378
2,Cash,15326
3,Credit Card,15238
4,Debit Card,15152
5,PayPal,15101


In [324]:
sql("""

SELECT

ShippingMethod,

COUNT(*) Orders,

AVG(Profit) AvgProfit

FROM fact_sales

GROUP BY ShippingMethod

""")

,ShippingMethod,Orders,AvgProfit
0,Overnight,30385,693.277495
1,Ground,30486,699.279489
2,2-Day,30797,684.504619


In [325]:
sql("""

SELECT

COUNT(DISTINCT r.OrderID) AS Returns,

COUNT(DISTINCT f.OrderID)

Orders,

ROUND(

COUNT(DISTINCT r.OrderID)*100.0

/

COUNT(DISTINCT f.OrderID),

2

)

ReturnRate

FROM fact_sales f

LEFT JOIN returns r

USING(OrderID)

""")

,Returns,Orders,ReturnRate
0,6418,91661,7.0


In [326]:
sql("""

SELECT

Category,

AVG(ProfitMargin)

Margin

FROM fact_sales

JOIN dim_product

USING(ProductID)

GROUP BY Category

ORDER BY Margin DESC

""")

,Category,Margin
0,Clothing,0.377051
1,Electronics,0.373915
2,Sports,0.371506
3,Furniture,0.361220
4,Home,0.355181


In [327]:
sql("""

SELECT

SUM(Sales) Revenue,

SUM(Profit) Profit,

AVG(Sales) AvgOrder,

COUNT(DISTINCT CustomerID) Customers,

COUNT(*) Orders,

SUM(Quantity) Units

FROM fact_sales

""")

,Revenue,Profit,AvgOrder,Customers,Orders,Units
0,1.690323e+08,6.346416e+07,1843.962353,9999,91668,328363.0


✓ CTEs

✓ Window Functions

✓ Ranking

✓ LAG()

✓ Running Totals

✓ Moving Average

✓ Pareto Analysis

✓ ABC Classification

✓ Business KPIs

✓ Segmentation

✓ Aggregations

✓ Analytical SQL

# Business Metrics & KPI Engine

This section creates reusable business metrics from the analytical
warehouse.

Rather than embedding calculations inside visualizations, we compute
them once and store them in reusable objects.

These KPIs will power:

- Executive Dashboard
- Data Quality Report
- Executive Summary
- Business Recommendations

In [328]:
from datetime import datetime

In [329]:
fact = fact_sales.merge(
    dim_date[["DateID", "Date"]],
    on="DateID",
    how="left"
)

fact.rename(columns={"Date": "OrderDate"}, inplace=True)
fact["OrderDate"] = pd.to_datetime(fact["OrderDate"])

In [330]:
fact = fact_sales.copy()


# ensure OrderDate exists; if not, populate from dim_date via DateID
if "OrderDate" not in fact.columns:
    fact = fact.merge(dim_date[["DateID", "Date"]], on="DateID", how="left")
    fact.rename(columns={"Date": "OrderDate"}, inplace=True)

fact["OrderDate"] = pd.to_datetime(fact["OrderDate"], errors="coerce")

Executive KPIs

In [331]:
total_revenue = fact["Sales"].sum()

In [332]:
total_profit = fact["Profit"].sum()

In [333]:
total_orders = len(fact)

In [334]:
total_customers = fact["CustomerID"].nunique()

In [335]:
total_units = fact["Quantity"].sum()

In [336]:
average_order_value = (

    fact.groupby("OrderID")["Sales"]

    .sum()

    .mean()

)

In [337]:
gross_margin = (

    total_profit /

    total_revenue

)

In [338]:
revenue_per_customer = (

    total_revenue /

    total_customers

)

In [339]:
revenue_per_store = (

    fact.groupby("StoreID")

    ["Sales"]

    .sum()

    .mean()

)

Growth Metrics

In [340]:
monthly_sales = (
    fact
    .groupby(
        pd.Grouper(
            key="OrderDate",
            freq="M"
        )
    )["Sales"]
    .sum()
    .reset_index()
)


In [341]:
monthly_sales["Growth"] = (

    monthly_sales["Sales"]

    .pct_change()

    *100

)

In [342]:
yearly_sales = (

    fact

    .groupby(

        fact.OrderDate.dt.year

    )

    ["Sales"]

    .sum()

    .reset_index()

)

Customer Metrics

In [343]:
repeat_customers = (

    fact

    .groupby("CustomerID")

    ["OrderID"]

    .count()

)

repeat_customer_count = (

    repeat_customers >= 2

).sum()

In [344]:
repeat_rate = (

    repeat_customer_count

    /

    total_customers

)

In [345]:
top_customer = (

    fact

    .groupby("CustomerID")

    ["Sales"]

    .sum()

    .idxmax()

)

In [346]:
average_clv = (

    dim_customer

    ["CustomerLifetimeValue"]

    .mean()

)

Product Metrics

In [347]:
top_product = (

    fact

    .groupby("ProductID")

    ["Sales"]

    .sum()

    .idxmax()

)

In [ ]:
category_sales = (

    fact

    .merge(

        dim_product,

        on="ProductID",

    )

    .groupby("Category")

    ["Sales"]

    .sum()

)

top_category = category_sales.idxmax()

In [ ]:
margin_category = (

    fact

    .merge(

        dim_product,

        on="ProductID",

    )

    .groupby("Category")

    ["ProfitMargin"]

    .mean()

)

highest_margin_category = margin_category.idxmax()

Store Metrics

In [350]:
best_store = (

    fact

    .groupby("StoreID")

    ["Sales"]

    .sum()

    .idxmax()

)

In [351]:
average_store_revenue = (

    fact

    .groupby("StoreID")

    ["Sales"]

    .sum()

    .mean()

)

Return Metrics

In [352]:
return_rate = (

    len(returns)

    /

    len(fact)

)

In [353]:
return_reason_summary = (

    returns

    .groupby("Reason")

    .size()

    .sort_values(

        ascending=False

    )

)

Payment Metrics

In [354]:
payment_summary = (

    fact

    .groupby("PaymentMethod")

    .size()

    .sort_values(

        ascending=False

    )

)

Shipping Metrics

In [355]:
shipping_summary = (

    fact

    .groupby("ShippingMethod")

    .agg(

        Orders=("OrderID","count"),

        Revenue=("Sales","sum"),

        Profit=("Profit","sum")

    )

)

KPI Dictionary

In [356]:
kpis = {

    "Revenue": total_revenue,

    "Profit": total_profit,

    "Orders": total_orders,

    "Customers": total_customers,

    "Units Sold": total_units,

    "Average Order Value": average_order_value,

    "Gross Margin": gross_margin,

    "Revenue Per Customer": revenue_per_customer,

    "Revenue Per Store": revenue_per_store,

    "Repeat Customer Rate": repeat_rate,

    "Average CLV": average_clv,

    "Return Rate": return_rate,

    "Top Category": top_category,

    "Highest Margin Category": highest_margin_category

}

KPI DataFrame

In [357]:
kpi_df = pd.DataFrame(

    kpis.items(),

    columns=[

        "Metric",

        "Value"

    ]

)

kpi_df

,Metric,Value
0,Revenue,169032340.94
1,Profit,63464159.94
2,Orders,91668
3,Customers,9999
4,Units Sold,328363
5,Average Order Value,1844.103173
6,Gross Margin,0.375456
7,Revenue Per Customer,16904.924586
8,Revenue Per Store,3380646.8188
9,Repeat Customer Rate,0.9993


Export KPI Report

In [358]:
kpi_df.to_csv(

    REPORT_PATH /

    "business_metrics.csv",

    index=False

)

KPI Summary

In [359]:
print("="*70)

print("EXECUTIVE KPI SUMMARY")

print("="*70)

print(f"Revenue               : ${total_revenue:,.2f}")

print(f"Profit                : ${total_profit:,.2f}")

print(f"Orders                : {total_orders:,}")

print(f"Customers             : {total_customers:,}")

print(f"Units Sold            : {total_units:,}")

print(f"Average Order Value   : ${average_order_value:,.2f}")

print(f"Gross Margin          : {gross_margin:.2%}")

print(f"Return Rate           : {return_rate:.2%}")

print("="*70)

EXECUTIVE KPI SUMMARY
Revenue               : $169,032,340.94
Profit                : $63,464,159.94
Orders                : 91,668
Customers             : 9,999
Units Sold            : 328,363
Average Order Value   : $1,844.10
Gross Margin          : 37.55%
Return Rate           : 7.64%


# Interactive Executive Dashboard

This dashboard provides an executive-level view of the retail business, combining KPI cards, trend charts, segment breakdowns, and operational visuals into one interactive summary.

In [360]:
# ==========================================================
# Dashboard Theme
# ==========================================================

import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots

px.defaults.template = "plotly_white"

In [361]:
def money(x):

    return "${:,.0f}".format(x)

Executive KPI Cards

In [362]:
fig = make_subplots(

    rows=2,

    cols=4,

    specs=[[

        {"type":"indicator"},

        {"type":"indicator"},

        {"type":"indicator"},

        {"type":"indicator"}

    ],

    [

        {"type":"indicator"},

        {"type":"indicator"},

        {"type":"indicator"},

        {"type":"indicator"}

    ]]

)

In [363]:
fig.add_trace(

    go.Indicator(

        mode="number",

        title={"text":"Revenue"},

        value=total_revenue,

        number={"prefix":"$"}

    ),

    row=1,

    col=1

)

fig.add_trace(

    go.Indicator(

        mode="number",

        title={"text":"Profit"},

        value=total_profit,

        number={"prefix":"$"}

    ),

    row=1,

    col=2

)

fig.add_trace(

    go.Indicator(

        mode="number",

        title={"text":"Orders"},

        value=total_orders

    ),

    row=1,

    col=3

)

fig.add_trace(

    go.Indicator(

        mode="number",

        title={"text":"Customers"},

        value=total_customers

    ),

    row=1,

    col=4

)

fig.add_trace(

    go.Indicator(

        mode="number",

        title={"text":"Average Order"},

        value=average_order_value,

        number={"prefix":"$"}

    ),

    row=2,

    col=1

)

fig.add_trace(

    go.Indicator(

        mode="number",

        title={"text":"Gross Margin"},

        value=gross_margin*100,

        number={"suffix":"%"}

    ),

    row=2,

    col=2

)

fig.add_trace(

    go.Indicator(

        mode="number",

        title={"text":"Return Rate"},

        value=return_rate*100,

        number={"suffix":"%"}

    ),

    row=2,

    col=3

)

fig.add_trace(

    go.Indicator(

        mode="number",

        title={"text":"Repeat Rate"},

        value=repeat_rate*100,

        number={"suffix":"%"}

    ),

    row=2,

    col=4

)

fig.update_layout(

    height=600,

    title="Executive KPI Dashboard"

)

fig.show()

In [364]:
fig = px.line(

    monthly_sales,

    x="OrderDate",

    y="Sales",

    title="Monthly Revenue Trend",

    markers=True

)

fig.show()

In [365]:
fig = px.bar(

    monthly_sales,

    x="OrderDate",

    y="Growth",

    title="Month-over-Month Growth"

)

fig.show()

In [366]:
category_dashboard = (

    fact

    .merge(

        dim_product,

        on="ProductID"

    )

    .groupby("Category")

    ["Sales"]

    .sum()

    .reset_index()

)

fig = px.bar(

    category_dashboard,

    x="Category",

    y="Sales",

    color="Sales",

    title="Revenue by Category"

)

fig.show()

In [401]:
profit_category = (

    orders_clean

    .merge(

        dim_product[["ProductID", "Category"]],

        on="ProductID",

        how="left",

    )

    .groupby("Category")

    ["Profit"]

    .sum()

    .reset_index()

)

px.bar(

    profit_category,

    x="Category",

    y="Profit",

    color="Profit",

    title="Profit by Category"

).show()

In [397]:
top_products = (

    orders_clean

    .merge(

        dim_product[["ProductID", "ProductName"]],

        on="ProductID",

        how="left",

    )

    .groupby("ProductName")

    ["Sales"]

    .sum()

    .nlargest(15)

    .reset_index()

)

px.bar(

    top_products,

    x="Sales",

    y="ProductName",

    orientation="h",

    title="Top Products"

).show()

In [398]:
top_customers = (

    orders_clean

    .merge(

        dim_customer[["CustomerID", "CustomerName"]],

        on="CustomerID",

        how="left",

    )

    .groupby("CustomerName")

    ["Sales"]

    .sum()

    .nlargest(20)

    .reset_index()

)

px.bar(

    top_customers,

    x="Sales",

    y="CustomerName",

    orientation="h",

    title="Top Customers"

).show()

In [399]:
store_dashboard = (

    orders_clean

    .merge(

        dim_store[["StoreID", "StoreName"]],

        on="StoreID",

        how="left",

    )

    .groupby("StoreName")

    ["Sales"]

    .sum()

    .reset_index()

)

px.bar(

    store_dashboard,

    x="StoreName",

    y="Sales",

    title="Sales by Store"

).show()

In [371]:
segments = (

    dim_customer

    .groupby("Segment")

    .size()

    .reset_index(name="Customers")

)

px.pie(

    segments,

    values="Customers",

    names="Segment",

    title="Customer Segments"

).show()

In [372]:
payment = (

    fact

    .groupby("PaymentMethod")

    .size()

    .reset_index(name="Orders")

)

px.pie(

    payment,

    values="Orders",

    names="PaymentMethod",

    title="Payment Method Distribution"

).show()

In [373]:
shipping = (

    fact

    .groupby("ShippingMethod")

    .size()

    .reset_index(name="Orders")

)

px.bar(

    shipping,

    x="ShippingMethod",

    y="Orders",

    title="Shipping Method Usage"

).show()

In [374]:
heat = fact.copy()

heat["Month"] = heat["OrderDate"].dt.month_name()

heat["Weekday"] = heat["OrderDate"].dt.day_name()

heat = (

    heat

    .groupby(

        [

            "Month",

            "Weekday"

        ]

    )

    ["Sales"]

    .sum()

    .reset_index()

)

fig = px.density_heatmap(

    heat,

    x="Month",

    y="Weekday",

    z="Sales",

    title="Sales Heatmap"

)

fig.show()

In [375]:
px.histogram(

    fact,

    x="ProfitMargin",

    nbins=40,

    title="Profit Margin Distribution"

).show()

In [376]:
px.box(

    fact,

    y="Sales",

    title="Sales Distribution"

).show()

In [377]:
px.scatter(

    fact.sample(5000),

    x="Sales",

    y="Profit",

    color="Quantity",

    size="Quantity",

    title="Sales vs Profit"

).show()

In [400]:
tree = (

    orders_clean

    .merge(

        dim_product[["ProductID", "Category", "ProductName"]],

        on="ProductID",

        how="left",

    )

    .groupby(

        [

            "Category",

            "ProductName"

        ]

    )

    ["Sales"]

    .sum()

    .reset_index()

)

px.treemap(

    tree,

    path=[

        "Category",

        "ProductName"

    ],

    values="Sales",

    title="Revenue Treemap"

).show()

In [379]:
px.area(

    monthly_sales,

    x="OrderDate",

    y="Sales",

    title="Revenue Area Chart"

).show()

In [380]:
monthly_sales["RunningRevenue"] = (

    monthly_sales["Sales"]

    .cumsum()

)

px.line(

    monthly_sales,

    x="OrderDate",

    y="RunningRevenue",

    title="Cumulative Revenue"

).show()

In [381]:
monthly_sales["Profit"] = (

    fact

    .groupby(

        pd.Grouper(

            key="OrderDate",

            freq="M"

        )

    )

    ["Profit"]

    .sum()

    .values

)

px.line(

    monthly_sales,

    x="OrderDate",

    y=[

        "Sales",

        "Profit"

    ],

    title="Revenue vs Profit"

).show()

In [382]:
print("="*60)

print("Interactive Dashboard Complete")

print("="*60)

print("20 Interactive Charts Created")

print("="*60)

Interactive Dashboard Complete
20 Interactive Charts Created


# Automated Reporting

The final stage of the pipeline produces executive-ready artifacts for stakeholders. These outputs summarize data quality, business metrics, insights, recommendations, and execution statistics.

In [383]:
# =====================================================
# Pipeline Statistics
# =====================================================

from pathlib import Path
from datetime import datetime

pipeline_end = datetime.now()

pipeline_start = datetime.now()
pipeline_duration = pipeline_end - pipeline_start

stats = {

    "Execution Time":str(pipeline_duration),

    "Raw Orders":len(orders),

    "Clean Orders":len(orders_clean),

    "Customers":len(dim_customer),

    "Products":len(dim_product),

    "Stores":len(dim_store),

    "Dates":len(dim_date)

}

stats

{'Execution Time': '-1 day, 23:59:59.999983',
 'Raw Orders': 100500,
 'Clean Orders': 91668,
 'Customers': 10000,
 'Products': 1000,
 'Stores': 50,
 'Dates': 731}

Data Quality Report

In [384]:
quality_report = pd.DataFrame({

    "Metric":[

        "Duplicate Orders",

        "Missing Customer IDs",

        "Missing Sales",

        "Negative Quantities",

        "Future Dates",

        "Profit Outliers",

        "Overall Quality Score"

    ],

    "Value":[

        validation_summary.loc[
            validation_summary.Check=="Duplicate Orders",
            "Issues"
        ].values[0],

        validation_summary.loc[
            validation_summary.Check=="Missing Customer IDs",
            "Issues"
        ].values[0],

        validation_summary.loc[
            validation_summary.Check=="Missing Sales",
            "Issues"
        ].values[0],

        validation_summary.loc[
            validation_summary.Check=="Negative Quantity",
            "Issues"
        ].values[0],

        validation_summary.loc[
            validation_summary.Check=="Future Dates",
            "Issues"
        ].values[0],

        validation_summary.loc[
            validation_summary.Check=="Profit Outliers",
            "Issues"
        ].values[0],

        quality_score

    ]

})

quality_report

,Metric,Value
0,Duplicate Orders,470.00
1,Missing Customer IDs,1005.00
2,Missing Sales,502.00
3,Negative Quantities,302.00
4,Future Dates,201.00
5,Profit Outliers,7355.00
6,Overall Quality Score,14.29


Executive KPI Report

In [385]:
executive_report = pd.DataFrame({

    "Metric":[

        "Revenue",

        "Profit",

        "Orders",

        "Customers",

        "Units Sold",

        "Average Order Value",

        "Gross Margin",

        "Return Rate",

        "Repeat Customer Rate"

    ],

    "Value":[

        total_revenue,

        total_profit,

        total_orders,

        total_customers,

        total_units,

        average_order_value,

        gross_margin,

        return_rate,

        repeat_rate

    ]

})

executive_report

,Metric,Value
0,Revenue,1.690323e+08
1,Profit,6.346416e+07
2,Orders,9.166800e+04
3,Customers,9.999000e+03
4,Units Sold,3.283630e+05
5,Average Order Value,1.844103e+03
6,Gross Margin,3.754557e-01
7,Return Rate,7.636253e-02
8,Repeat Customer Rate,9.992999e-01


Automatically Generate Insights

In [386]:
insights = []

if gross_margin > 0.30:
    insights.append(
        "Gross margin exceeds 30%, indicating healthy profitability."
    )

if return_rate > 0.08:
    insights.append(
        "Return rate is above target and should be investigated."
    )
else:
    insights.append(
        "Return rate remains within acceptable limits."
    )

if repeat_rate > 0.40:
    insights.append(
        "Customer retention is strong with a healthy repeat purchase rate."
    )
else:
    insights.append(
        "Opportunity exists to improve customer retention."
    )

if monthly_sales["Growth"].mean() > 0:
    insights.append(
        "Overall monthly revenue trend is positive."
    )

if total_profit > 0:
    insights.append(
        "The business generated positive profit during the reporting period."
    )

insights

['Gross margin exceeds 30%, indicating healthy profitability.',
 'Return rate remains within acceptable limits.',
 'Customer retention is strong with a healthy repeat purchase rate.',
 'Overall monthly revenue trend is positive.',
 'The business generated positive profit during the reporting period.']

Recommendations

In [387]:
recommendations = [

    "Increase marketing investment in the highest-performing product categories.",

    "Target repeat customers with loyalty and rewards programs.",

    "Investigate products with above-average return rates.",

    "Review inventory levels for top-selling products.",

    "Expand successful stores and optimize underperforming locations.",

    "Monitor gross margin monthly to maintain profitability.",

    "Continue tracking KPIs through the DuckDB warehouse.",

    "Schedule automated data quality checks before each warehouse refresh."

]

recommendation_df = pd.DataFrame({

    "Recommendation":recommendations

})

recommendation_df

,Recommendation
0,Increase marketing investment in the highest-p...
1,Target repeat customers with loyalty and rewar...
2,Investigate products with above-average return...
3,Review inventory levels for top-selling products.
4,Expand successful stores and optimize underper...
5,Monitor gross margin monthly to maintain profi...
6,Continue tracking KPIs through the DuckDB ware...
7,Schedule automated data quality checks before ...


Export Reports

In [388]:
quality_report.to_csv(
    REPORT_PATH/"data_quality_report.csv",
    index=False
)

executive_report.to_csv(
    REPORT_PATH/"executive_kpi_report.csv",
    index=False
)

recommendation_df.to_csv(
    REPORT_PATH/"recommendations.csv",
    index=False
)

audit_df.to_csv(
    REPORT_PATH/"cleaning_audit.csv",
    index=False
)

profile_df.to_csv(
    REPORT_PATH/"dataset_profile.csv",
    index=False
)

print("All reports exported.")

All reports exported.


Final Executive Summary

In [389]:
summary = f"""
==========================================================
EXECUTIVE SUMMARY
==========================================================

Revenue Generated

${total_revenue:,.2f}

----------------------------------------------------------

Profit Generated

${total_profit:,.2f}

----------------------------------------------------------

Orders Processed

{total_orders:,}

----------------------------------------------------------

Customers

{total_customers:,}

----------------------------------------------------------

Average Order Value

${average_order_value:,.2f}

----------------------------------------------------------

Gross Margin

{gross_margin:.2%}

----------------------------------------------------------

Repeat Customer Rate

{repeat_rate:.2%}

----------------------------------------------------------

Return Rate

{return_rate:.2%}

==========================================================
"""

print(summary)


EXECUTIVE SUMMARY

Revenue Generated

$169,032,340.94

----------------------------------------------------------

Profit Generated

$63,464,159.94

----------------------------------------------------------

Orders Processed

91,668

----------------------------------------------------------

Customers

9,999

----------------------------------------------------------

Average Order Value

$1,844.10

----------------------------------------------------------

Gross Margin

37.55%

----------------------------------------------------------

Repeat Customer Rate

99.93%

----------------------------------------------------------

Return Rate

7.64%




Project Deliverables

In [390]:
deliverables = [

    "raw/customers.csv",

    "raw/products.csv",

    "raw/stores.csv",

    "raw/orders.csv",

    "raw/returns.csv",

    "clean/customers.parquet",

    "clean/products.parquet",

    "clean/stores.parquet",

    "clean/orders.parquet",

    "warehouse/fact_sales.parquet",

    "warehouse/dim_customer.parquet",

    "warehouse/dim_product.parquet",

    "warehouse/dim_store.parquet",

    "warehouse/dim_date.parquet",

    "warehouse/retail_dw.duckdb",

    "reports/business_metrics.csv",

    "reports/data_quality_report.csv",

    "reports/executive_kpi_report.csv",

    "reports/recommendations.csv",

    "reports/cleaning_audit.csv",

    "reports/dataset_profile.csv"

]

deliverables_df = pd.DataFrame({

    "Generated Files":deliverables

})

deliverables_df

,Generated Files
0,raw/customers.csv
1,raw/products.csv
2,raw/stores.csv
3,raw/orders.csv
4,raw/returns.csv
5,clean/customers.parquet
6,clean/products.parquet
7,clean/stores.parquet
8,clean/orders.parquet
9,warehouse/fact_sales.parquet


Pipeline Complete

In [391]:
print("=" * 70)
print("END-TO-END DATA ENGINEERING PIPELINE COMPLETE")
print("=" * 70)

print("✓ Fake Data Generated with Faker")
print("✓ CSV Files Created")
print("✓ Data Validation Completed")
print("✓ Data Cleaning Pipeline Executed")
print("✓ Feature Engineering Completed")
print("✓ Star Schema Created")
print("✓ DuckDB Warehouse Built")
print("✓ Advanced SQL Analytics Executed")
print("✓ KPI Engine Generated")
print("✓ Interactive Dashboard Created")
print("✓ Data Quality Report Exported")
print("✓ Executive Summary Generated")

print("=" * 70)

END-TO-END DATA ENGINEERING PIPELINE COMPLETE
✓ Fake Data Generated with Faker
✓ CSV Files Created
✓ Data Validation Completed
✓ Data Cleaning Pipeline Executed
✓ Feature Engineering Completed
✓ Star Schema Created
✓ DuckDB Warehouse Built
✓ Advanced SQL Analytics Executed
✓ KPI Engine Generated
✓ Interactive Dashboard Created
✓ Data Quality Report Exported
✓ Executive Summary Generated
